In [1]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

print("Imports loaded.")

Imports loaded.


In [2]:
def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(5):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found. Run Jupyter from project root or keep /data and /notebooks in place.")

NOTEBOOK_PATH = Path.cwd()

PROJECT_ROOT = find_project_root(NOTEBOOK_PATH)

RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

behavioural_file = PROCESSED_DIR / "AIS_2024_01_panynj_behavioural.parquet"

print("Loading scoped dataset...")
print("File exists:", behavioural_file.exists())

AIS_behavioural = pd.read_parquet(behavioural_file)

print("\nDataset loaded.")
display(AIS_behavioural.shape)
display(AIS_behavioural.head())

Loading scoped dataset...
File exists: True

Dataset loaded.


(10854660, 27)

,MMSI,Day,TimeOfDay,LAT,LON,SOG,COG,Heading,Status,Timestamp,...,acceleration_mps2,movement_state,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,209697000,10,22:00:54,27.38801,-94.12006,12.2,316.0,313.0,0.0,2024-01-10 22:00:54,...,NaN,UNKNOWN,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
1,209697000,10,22:19:15,27.43232,-94.16890,12.2,316.0,314.0,0.0,2024-01-10 22:19:15,...,NaN,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
2,209697000,10,22:23:56,27.44364,-94.18141,12.2,315.2,312.0,0.0,2024-01-10 22:23:56,...,0.000048,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
3,209697000,10,22:25:05,27.44647,-94.18452,12.2,316.0,312.0,0.0,2024-01-10 22:25:05,...,0.001391,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
4,209697000,10,22:46:15,27.50442,-94.23301,12.3,323.0,319.0,0.0,2024-01-10 22:46:15,...,-0.000040,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A


Fact Table Grain Definition

One row in the fact table represents:

A single vessel position observation for a specific vessel at a specific timestamp in the PANYNJ region.

In [3]:
AIS_behavioural = AIS_behavioural.reset_index(drop=True)

AIS_behavioural["event_id"] = (
    AIS_behavioural.index + 1
)
print("Unique event_id check:", AIS_behavioural["event_id"].is_unique)
display(AIS_behavioural.head())

Unique event_id check: True


,MMSI,Day,TimeOfDay,LAT,LON,SOG,COG,Heading,Status,Timestamp,...,movement_state,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,event_id
0,209697000,10,22:00:54,27.38801,-94.12006,12.2,316.0,313.0,0.0,2024-01-10 22:00:54,...,UNKNOWN,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A,1
1,209697000,10,22:19:15,27.43232,-94.16890,12.2,316.0,314.0,0.0,2024-01-10 22:19:15,...,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A,2
2,209697000,10,22:23:56,27.44364,-94.18141,12.2,315.2,312.0,0.0,2024-01-10 22:23:56,...,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A,3
3,209697000,10,22:25:05,27.44647,-94.18452,12.2,316.0,312.0,0.0,2024-01-10 22:25:05,...,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A,4
4,209697000,10,22:46:15,27.50442,-94.23301,12.3,323.0,319.0,0.0,2024-01-10 22:46:15,...,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A,5


In [4]:
cols = ["event_id"] + [
    c for c in AIS_behavioural.columns
    if c != "event_id"
]

AIS_behavioural = AIS_behavioural[cols]
display(AIS_behavioural.head())

,event_id,MMSI,Day,TimeOfDay,LAT,LON,SOG,COG,Heading,Status,...,acceleration_mps2,movement_state,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,1,209697000,10,22:00:54,27.38801,-94.12006,12.2,316.0,313.0,0.0,...,NaN,UNKNOWN,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
1,2,209697000,10,22:19:15,27.43232,-94.16890,12.2,316.0,314.0,0.0,...,NaN,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
2,3,209697000,10,22:23:56,27.44364,-94.18141,12.2,315.2,312.0,0.0,...,0.000048,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
3,4,209697000,10,22:25:05,27.44647,-94.18452,12.2,316.0,312.0,0.0,...,0.001391,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
4,5,209697000,10,22:46:15,27.50442,-94.23301,12.3,323.0,319.0,0.0,...,-0.000040,NORMAL,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A


In [5]:
vessel_cols = [
    "MMSI",
    "VesselName",
    "CallSign",
    "IMO",
    "VesselType",
    "Length",
    "Width",
    "Draft",
    "TransceiverClass"
]

dim_vessel = (
    AIS_behavioural[vessel_cols]
    .drop_duplicates(subset=["MMSI"])
    .reset_index(drop=True)
)
display(dim_vessel.head())
display(dim_vessel.shape)

,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,209697000,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
1,209850000,ANDROMEDA J,5BKR3,IMO9355422,70,140.0,22.0,5.6,A
2,209862000,WARNOW WHALE,5BXD3,IMO9395032,70,166.0,25.0,6.9,A
3,210749000,PACIFIC TRADER,5BPJ3,IMO9406922,74,147.0,23.0,7.7,A
4,210905000,ARSOS,5BAQ2,IMO9395123,70,166.0,25.0,7.4,A


(868, 9)

In [6]:
dim_vessel["vessel_id"] = dim_vessel.index + 1

In [7]:
cols = ["vessel_id"] + [c for c in dim_vessel.columns if c != "vessel_id"]
dim_vessel = dim_vessel[cols]

In [8]:
print("Unique MMSI:", dim_vessel["MMSI"].is_unique)
print("Unique vessel_id:", dim_vessel["vessel_id"].is_unique)
display(dim_vessel.head())
display(dim_vessel.shape)

Unique MMSI: True
Unique vessel_id: True


,vessel_id,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,1,209697000,CONTSHIP LEO,5BHD6,IMO9403451,74,148.0,25.0,8.3,A
1,2,209850000,ANDROMEDA J,5BKR3,IMO9355422,70,140.0,22.0,5.6,A
2,3,209862000,WARNOW WHALE,5BXD3,IMO9395032,70,166.0,25.0,6.9,A
3,4,210749000,PACIFIC TRADER,5BPJ3,IMO9406922,74,147.0,23.0,7.7,A
4,5,210905000,ARSOS,5BAQ2,IMO9395123,70,166.0,25.0,7.4,A


(868, 10)

In [9]:
display(AIS_behavioural.columns.tolist())

['event_id',
 'MMSI',
 'Day',
 'TimeOfDay',
 'LAT',
 'LON',
 'SOG',
 'COG',
 'Heading',
 'Status',
 'Timestamp',
 'prev_timestamp',
 'prev_LAT',
 'prev_LON',
 'time_delta_seconds',
 'distance_m',
 'speed_mps',
 'prev_speed_mps',
 'acceleration_mps2',
 'movement_state',
 'VesselName',
 'CallSign',
 'IMO',
 'VesselType',
 'Length',
 'Width',
 'Draft',
 'TransceiverClass']

In [10]:
display(AIS_behavioural["Timestamp"])

0          2024-01-10 22:00:54
1          2024-01-10 22:19:15
2          2024-01-10 22:23:56
3          2024-01-10 22:25:05
4          2024-01-10 22:46:15
                   ...        
10854655   2024-01-01 17:13:04
10854656   2024-01-01 17:14:07
10854657   2024-01-01 17:15:21
10854658   2024-01-01 17:16:30
10854659   2024-01-01 17:18:31
Name: Timestamp, Length: 10854660, dtype: datetime64[us]

In [11]:
dim_date = (
    AIS_behavioural[["Timestamp"]]
    .copy()
)

dim_date["date"] = dim_date["Timestamp"].dt.date

dim_date = (
    dim_date[["date"]]
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

In [12]:
dim_date["year"] = pd.to_datetime(dim_date["date"]).dt.year
dim_date["month"] = pd.to_datetime(dim_date["date"]).dt.month
dim_date["month_name"] = pd.to_datetime(dim_date["date"]).dt.month_name()

dim_date["day"] = pd.to_datetime(dim_date["date"]).dt.day
dim_date["day_of_week"] = pd.to_datetime(dim_date["date"]).dt.dayofweek
dim_date["day_name"] = pd.to_datetime(dim_date["date"]).dt.day_name()

dim_date["week"] = pd.to_datetime(dim_date["date"]).dt.isocalendar().week
dim_date["is_weekend"] = dim_date["day_of_week"] >= 5

In [13]:
dim_date["date_id"] = (
    pd.to_datetime(dim_date["date"])
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [14]:
cols = ["date_id"] + [c for c in dim_date.columns if c != "date_id"]
dim_date = dim_date[cols]

In [15]:
print("Unique date_id:", dim_date["date_id"].is_unique)
print("Rows:", dim_date.shape[0])
display(dim_date)

Unique date_id: True
Rows: 31


,date_id,date,year,month,month_name,day,day_of_week,day_name,week,is_weekend
0,20240101,2024-01-01,2024,1,January,1,0,Monday,1,False
1,20240102,2024-01-02,2024,1,January,2,1,Tuesday,1,False
2,20240103,2024-01-03,2024,1,January,3,2,Wednesday,1,False
3,20240104,2024-01-04,2024,1,January,4,3,Thursday,1,False
4,20240105,2024-01-05,2024,1,January,5,4,Friday,1,False
5,20240106,2024-01-06,2024,1,January,6,5,Saturday,1,True
6,20240107,2024-01-07,2024,1,January,7,6,Sunday,1,True
7,20240108,2024-01-08,2024,1,January,8,0,Monday,2,False
8,20240109,2024-01-09,2024,1,January,9,1,Tuesday,2,False
9,20240110,2024-01-10,2024,1,January,10,2,Wednesday,2,False


In [16]:
AIS_behavioural["lat_grid"] = AIS_behavioural["LAT"].round(3)
AIS_behavioural["lon_grid"] = AIS_behavioural["LON"].round(3)

In [17]:
dim_location = (
    AIS_behavioural[["lat_grid", "lon_grid"]]
    .drop_duplicates()
    .sort_values(["lat_grid", "lon_grid"])
    .reset_index(drop=True)
)

In [18]:
dim_location["location_id"] = dim_location.index + 1

In [19]:
dim_location["grid_label"] = (
    dim_location["lat_grid"].astype(str)
    + ", "
    + dim_location["lon_grid"].astype(str)
)

In [20]:
cols = ["location_id", "lat_grid", "lon_grid", "grid_label"]
dim_location = dim_location[cols]

In [21]:
print("Unique location_id:", dim_location["location_id"].is_unique)
print("Unique grid cells:", dim_location[["lat_grid", "lon_grid"]].duplicated().sum() == 0)
print("Rows:", dim_location.shape[0])
display(dim_location.head(20))

Unique location_id: True
Unique grid cells: True
Rows: 1665532


,location_id,lat_grid,lon_grid,grid_label
0,1,0.067,-74.254,"0.067, -74.254"
1,2,0.067,-74.250,"0.067, -74.25"
2,3,0.067,-74.233,"0.067, -74.233"
3,4,0.067,-74.200,"0.067, -74.2"
4,5,0.067,-74.128,"0.067, -74.128"
5,6,0.641,-74.129,"0.641, -74.129"
6,7,0.667,-74.254,"0.667, -74.254"
7,8,0.667,-74.253,"0.667, -74.253"
8,9,0.667,-74.205,"0.667, -74.205"
9,10,0.667,-74.203,"0.667, -74.203"


In [22]:
dim_movement_state = (
    AIS_behavioural[["movement_state"]]
    .drop_duplicates()
    .sort_values("movement_state")
    .reset_index(drop=True)
)

dim_movement_state["movement_state_id"] = dim_movement_state.index + 1

cols = ["movement_state_id", "movement_state"]
dim_movement_state = dim_movement_state[cols]

In [23]:
print("Unique movement_state_id:", dim_movement_state["movement_state_id"].is_unique)
print("Unique movement_state:", dim_movement_state["movement_state"].is_unique)
print("Rows:", dim_movement_state.shape[0])

display(dim_movement_state)

Unique movement_state_id: True
Unique movement_state: True
Rows: 6


,movement_state_id,movement_state
0,1,HIGH_SPEED
1,2,NORMAL
2,3,SLOW
3,4,STATIONARY
4,5,UNKNOWN
5,6,VERY_SLOW


In [24]:
AIS_modelled = AIS_behavioural.copy()

AIS_modelled = AIS_modelled.merge(
    dim_vessel[["vessel_id", "MMSI"]],
    on="MMSI",
    how="left",
    validate="many_to_one"
)

In [25]:
print("Original rows:", AIS_behavioural.shape[0])
print("Modelled rows:", AIS_modelled.shape[0])

print("Missing vessel_id:", AIS_modelled["vessel_id"].isna().sum())
print("Unique vessel_id:", AIS_modelled["vessel_id"].nunique())

Original rows: 10854660
Modelled rows: 10854660
Missing vessel_id: 0
Unique vessel_id: 868


In [26]:
AIS_modelled["date"] = AIS_modelled["Timestamp"].dt.date

AIS_modelled = AIS_modelled.merge(
    dim_date[["date_id", "date"]],
    on="date",
    how="left",
    validate="many_to_one"
)

In [27]:
print("Rows after date merge:", AIS_modelled.shape[0])
print("Missing date_id:", AIS_modelled["date_id"].isna().sum())
print("Unique date_id:", AIS_modelled["date_id"].nunique())

Rows after date merge: 10854660
Missing date_id: 0
Unique date_id: 31


In [28]:
AIS_modelled = AIS_modelled.merge(
    dim_location[["location_id", "lat_grid", "lon_grid"]],
    on=["lat_grid", "lon_grid"],
    how="left",
    validate="many_to_one"
)

In [29]:
print("Rows after location merge:", AIS_modelled.shape[0])
print("Missing location_id:", AIS_modelled["location_id"].isna().sum())
print("Unique location_id:", AIS_modelled["location_id"].nunique())

Rows after location merge: 10854660
Missing location_id: 0
Unique location_id: 1665532


In [30]:
AIS_modelled = AIS_modelled.merge(
    dim_movement_state[["movement_state_id", "movement_state"]],
    on="movement_state",
    how="left",
    validate="many_to_one"
)

In [31]:
print("Rows after movement state merge:", AIS_modelled.shape[0])
print("Missing movement_state_id:", AIS_modelled["movement_state_id"].isna().sum())
print("Unique movement_state_id:", AIS_modelled["movement_state_id"].nunique())

Rows after movement state merge: 10854660
Missing movement_state_id: 0
Unique movement_state_id: 6


In [32]:
fact_vessel_events = AIS_modelled[
    [
        "event_id",
        "vessel_id",
        "date_id",
        "location_id",
        "movement_state_id",
        "Timestamp",
        "LAT",
        "LON",
        "SOG",
        "COG",
        "Heading",
        "Status",
        "distance_m",
        "time_delta_seconds",
        "speed_mps",
        "prev_speed_mps",
        "acceleration_mps2"
    ]
].copy()

In [33]:
print("Fact rows:", fact_vessel_events.shape[0])
print("Original rows:", AIS_behavioural.shape[0])

print("Unique event_id:", fact_vessel_events["event_id"].is_unique)

print("Missing vessel_id:", fact_vessel_events["vessel_id"].isna().sum())
print("Missing date_id:", fact_vessel_events["date_id"].isna().sum())
print("Missing location_id:", fact_vessel_events["location_id"].isna().sum())
print("Missing movement_state_id:", fact_vessel_events["movement_state_id"].isna().sum())

Fact rows: 10854660
Original rows: 10854660
Unique event_id: True
Missing vessel_id: 0
Missing date_id: 0
Missing location_id: 0
Missing movement_state_id: 0


In [ ]:
dim_vessel.to_parquet(
    PROCESSED_DIR / "dim_vessel.parquet",
    index=False
)

dim_date.to_parquet(
    PROCESSED_DIR / "dim_date.parquet",
    index=False
)

dim_location.to_parquet(
    PROCESSED_DIR / "dim_location.parquet",
    index=False
)

dim_movement_state.to_parquet(
    PROCESSED_DIR / "dim_movement_state.parquet",
    index=False
)

fact_vessel_events.to_parquet(
    PROCESSED_DIR / "fact_vessel_events.parquet",
    index=False
)

print("Model tables exported successfully.")

In [ ]:
model_outputs = [
    "dim_vessel.parquet",
    "dim_date.parquet",
    "dim_location.parquet",
    "dim_movement_state.parquet",
    "fact_vessel_events.parquet"
]

for file in model_outputs:
    path = PROCESSED_DIR / file
    print(file, "exists:", path.exists())

In [ ]:
print("Final model table summary\n")

print("dim_vessel rows:", dim_vessel.shape[0])
print("dim_date rows:", dim_date.shape[0])
print("dim_location rows:", dim_location.shape[0])
print("dim_movement_state rows:", dim_movement_state.shape[0])
print("fact_vessel_events rows:", fact_vessel_events.shape[0])

print("\nFact table validation")
print("Fact rows match behavioural rows:", fact_vessel_events.shape[0] == AIS_behavioural.shape[0])
print("Unique event_id:", fact_vessel_events["event_id"].is_unique)

print("\nMissing foreign keys")
print("Missing vessel_id:", fact_vessel_events["vessel_id"].isna().sum())
print("Missing date_id:", fact_vessel_events["date_id"].isna().sum())
print("Missing location_id:", fact_vessel_events["location_id"].isna().sum())
print("Missing movement_state_id:", fact_vessel_events["movement_state_id"].isna().sum())